# 01. Feature Engineering Basics
### Notebook 1 – Introduction to Feature Engineering

**Dataset:** Online Retail Transactions (`data.csv`)
541,909 rows × 8 columns — InvoiceNo, StockCode, Description, Quantity, InvoiceDate, UnitPrice, CustomerID, Country


In [1]:
import pandas as pd
import numpy as np
pd.set_option('display.max_columns', None)
df = pd.read_csv('data.csv', encoding='ISO-8859-1')
print("Shape:", df.shape)
df.head()

Shape: (541909, 8)


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,12/1/2010 8:26,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,12/1/2010 8:26,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,12/1/2010 8:26,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,12/1/2010 8:26,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,12/1/2010 8:26,3.39,17850.0,United Kingdom


In [2]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   InvoiceNo    541909 non-null  str    
 1   StockCode    541909 non-null  str    
 2   Description  540455 non-null  str    
 3   Quantity     541909 non-null  int64  
 4   InvoiceDate  541909 non-null  str    
 5   UnitPrice    541909 non-null  float64
 6   CustomerID   406829 non-null  float64
 7   Country      541909 non-null  str    
dtypes: float64(2), int64(1), str(5)
memory usage: 67.3 MB


In [3]:
df.isnull().sum()

InvoiceNo           0
StockCode           0
Description      1454
Quantity            0
InvoiceDate         0
UnitPrice           0
CustomerID     135080
Country             0
dtype: int64

## 1. What is a Feature?

A **feature** is an individual measurable property or characteristic of the data used as input to a machine learning model.

In our dataset, each column is a *potential* feature:
- `Quantity` — how many units were bought
- `UnitPrice` — price per unit
- `InvoiceDate` — when the transaction occurred
- `Country` — where the customer is from

But not every column is automatically a *useful* feature — some need transformation first, and some (like `InvoiceNo`, a transaction ID) carry no predictive signal at all.

In [4]:
df.columns.tolist()

['InvoiceNo',
 'StockCode',
 'Description',
 'Quantity',
 'InvoiceDate',
 'UnitPrice',
 'CustomerID',
 'Country']

## 2. Feature vs Variable

The terms are often used interchangeably, but there's a subtle distinction:

| Term | Meaning |
|---|---|
| **Variable** | Any column in the raw dataset — statistical/data terminology |
| **Feature** | A variable (or something derived from it) that is *actually fed into* a model |

For example, `InvoiceDate` is a **variable** in our raw data. It's a raw timestamp — not yet usable by a model. Once we extract `Hour`, `DayOfWeek`, or `Month` from it, those become **features**.

So: every feature starts life as a variable, but not every variable becomes a feature as-is.

In [5]:
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])
df['InvoiceHour'] = df['InvoiceDate'].dt.hour
df['InvoiceDayOfWeek'] = df['InvoiceDate'].dt.day_name()
df[['InvoiceDate', 'InvoiceHour', 'InvoiceDayOfWeek']].head()

,InvoiceDate,InvoiceHour,InvoiceDayOfWeek
0,2010-12-01 08:26:00,8,Wednesday
1,2010-12-01 08:26:00,8,Wednesday
2,2010-12-01 08:26:00,8,Wednesday
3,2010-12-01 08:26:00,8,Wednesday
4,2010-12-01 08:26:00,8,Wednesday


## 3. Feature vs Target

- **Features (X)**: the inputs the model uses to learn patterns (e.g. Quantity, UnitPrice, Country, InvoiceHour)
- **Target (y)**: the output/label the model is trying to predict

Our raw dataset doesn't come with an explicit target column — it's transactional log data. But depending on the business question, we could *engineer* a target:

- Predicting UnitPrice → regression target
- Predicting whether a customer will churn → classification target (needs to be engineered from repeat purchase behavior)
- Predicting total spend per transaction → Quantity * UnitPrice (engineered target)

This is an important point: **sometimes the target itself must be engineered**, not just the features.

In [6]:
df['TotalPrice'] = df['Quantity'] * df['UnitPrice']
df[['Quantity', 'UnitPrice', 'TotalPrice']].head()

,Quantity,UnitPrice,TotalPrice
0,6,2.55,15.30
1,6,3.39,20.34
2,8,2.75,22.00
3,6,3.39,20.34
4,6,3.39,20.34


## 4. What is Feature Engineering?

**Feature Engineering** is the process of using domain knowledge and data manipulation techniques to create, transform, or select variables (features) that make machine learning models perform better.

It sits between raw data and model training:

Raw Data → Feature Engineering → Model-Ready Features → ML Model → Predictions

It includes activities like:
- Creating new features from existing ones (TotalPrice above)
- Transforming skewed or raw features (scaling, encoding, log transforms)
- Selecting the most relevant features
- Extracting structure from unstructured/complex fields (e.g. text, dates)

Feature engineering is often said to matter **more than the choice of algorithm** — a simple model with great features usually beats a complex model with poor features.

## 5. Why Feature Engineering?

1. **Raw data is rarely model-ready** — InvoiceDate as a string is useless to most algorithms until decomposed.
2. **Models can't infer relationships you don't give them** — a model can't know TotalPrice matters unless you compute it.
3. **Reduces noise and dimensionality** — removing irrelevant columns like InvoiceNo improves generalization.
4. **Improves model performance** — well-engineered features often yield bigger performance gains than hyperparameter tuning.
5. **Handles real-world messiness** — missing `CustomerID`, inconsistent Description text, negative Quantity (returns) all need handling.

In [7]:
print("Negative quantities (likely returns):", (df['Quantity'] < 0).sum())
print("Missing CustomerID:", df['CustomerID'].isnull().sum())
print("Zero or negative UnitPrice:", (df['UnitPrice'] <= 0).sum())

Negative quantities (likely returns): 10624
Missing CustomerID: 135080
Zero or negative UnitPrice: 2517


## 6. Importance of Domain Knowledge

Feature engineering is not a purely mechanical process — it relies heavily on **understanding the domain**.

In retail transaction data:
- A **negative Quantity** isn't dirty data — it represents a **product return**. A retail domain expert would engineer an IsReturn flag rather than dropping these rows.
- StockCode values like POST, DOT, or M aren't real products — they represent postage, dot charges, or manual entries. Domain knowledge tells us to filter or flag these separately.
- Knowing that retail sales spike around holidays means engineering IsHoliday or Month features could be valuable.

Without domain knowledge, you might "clean" away meaningful signal or engineer irrelevant features.

In [8]:
df['IsReturn'] = df['Quantity'] < 0
df['IsReturn'].value_counts()

IsReturn
False    531285
True      10624
Name: count, dtype: int64

In [9]:
df[df['StockCode'].isin(['POST', 'DOT', 'M'])]['StockCode'].value_counts()

StockCode
POST    1256
DOT      710
M        571
Name: count, dtype: int64

## 7. Good Features vs Bad Features

**Good features are:**
- Relevant to the target/task
- Not heavily correlated/redundant with other features
- Consistent and reliably available at prediction time
- Interpretable, where possible

**Bad features are:**
- Identifiers with no real signal (InvoiceNo, CustomerID as a raw number)
- Leaky (contain information from the future/target — see Section 13)
- Mostly missing or constant
- Redundant duplicates of another feature

| Column | Good or Bad (as-is)? | Why |
|---|---|---|
| `UnitPrice` | Good | Direct numeric signal |
| `InvoiceNo` | Bad | Just a transaction ID, no predictive value |
| `Description` | Bad (as raw text) | Needs extraction/encoding first |
| `TotalPrice` (engineered) | Good | Captures combined effect of Quantity & Price |

In [10]:
df.nunique()

InvoiceNo           25900
StockCode            4070
Description          4223
Quantity              722
InvoiceDate         23260
UnitPrice            1630
CustomerID           4372
Country                38
InvoiceHour            15
InvoiceDayOfWeek        6
TotalPrice           6204
IsReturn                2
dtype: int64

## 8. Feature Engineering Workflow

A typical feature engineering pipeline follows these stages:

1. **Understand the data & problem** (EDA, domain context)
2. **Handle missing values & inconsistencies**
3. **Feature Creation** — derive new features
4. **Feature Transformation** — scale, encode, normalize
5. **Feature Extraction** — pull structure from complex fields (text, dates, images)
6. **Feature Selection** — keep only the most useful features
7. **Validate** — check for leakage, test impact on model performance

This is iterative, not linear — you often loop back after seeing model results.

## 9. Feature Creation

Feature creation means deriving **new** features from existing raw columns, using arithmetic, domain logic, or aggregation.

Examples from our dataset:

In [11]:
# Example 1: Revenue per transaction line
df['TotalPrice'] = df['Quantity'] * df['UnitPrice']

# Example 2: Time-based features from InvoiceDate
df['InvoiceMonth'] = df['InvoiceDate'].dt.month
df['InvoiceYear'] = df['InvoiceDate'].dt.year

# Example 3: Customer-level aggregation (creating features from grouped data)
customer_spend = df.groupby('CustomerID')['TotalPrice'].sum().rename('CustomerTotalSpend')

df = df.merge(customer_spend, on='CustomerID', how='left')
df[['CustomerID', 'TotalPrice', 'CustomerTotalSpend']].head()

,CustomerID,TotalPrice,CustomerTotalSpend
0,17850.0,15.30,5288.63
1,17850.0,20.34,5288.63
2,17850.0,22.00,5288.63
3,17850.0,20.34,5288.63
4,17850.0,20.34,5288.63


## 10. Feature Transformation

Transformation changes the **representation** of a feature without necessarily creating new information — e.g., scaling, encoding, or fixing skew.

Examples:
- **Encoding**: converting Country (categorical) into numeric form
- **Scaling**: normalizing UnitPrice and Quantity so they're on comparable scales
- **Log transform**: TotalPrice is likely right-skewed (few very large transactions) — a log transform compresses that skew

In [12]:
top_countries = df['Country'].value_counts().nlargest(5).index
df['Country_grouped'] = np.where(df['Country'].isin(top_countries), df['Country'], 'Other')
country_encoded = pd.get_dummies(df['Country_grouped'], prefix='Country')
country_encoded.head()

,Country_EIRE,Country_France,Country_Germany,Country_Other,Country_Spain,Country_United Kingdom
0,False,False,False,False,False,True
1,False,False,False,False,False,True
2,False,False,False,False,False,True
3,False,False,False,False,False,True
4,False,False,False,False,False,True


In [13]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
df[['UnitPrice_scaled', 'Quantity_scaled']] = scaler.fit_transform(df[['UnitPrice', 'Quantity']])
df[['UnitPrice', 'UnitPrice_scaled', 'Quantity', 'Quantity_scaled']].head()

,UnitPrice,UnitPrice_scaled,Quantity,Quantity_scaled
0,2.55,-0.021301,6,-0.016289
1,3.39,-0.012620,6,-0.016289
2,2.75,-0.019234,8,-0.007118
3,3.39,-0.012620,6,-0.016289
4,3.39,-0.012620,6,-0.016289


In [14]:
df['TotalPrice_log'] = np.log1p(df['TotalPrice'].clip(lower=0))
df[['TotalPrice', 'TotalPrice_log']].describe()

,TotalPrice,TotalPrice_log
count,541909.000000,541909.000000
mean,17.987795,2.280430
std,378.810824,1.063879
min,-168469.600000,0.000000
25%,3.400000,1.481605
50%,9.750000,2.374906
75%,17.400000,2.912351
max,168469.600000,12.034517


## 11. Feature Selection

Feature selection means **choosing a subset** of relevant features and discarding redundant, irrelevant, or harmful ones — reducing dimensionality and overfitting risk.

Common approaches:
- **Filter methods**: correlation, chi-square, variance threshold
- **Wrapper methods**: recursive feature elimination (RFE)
- **Embedded methods**: feature importance from tree-based models, L1 regularization

We won't implement full selection yet (that's a later notebook) — just a quick correlation preview:

In [15]:
numeric_cols = ['Quantity', 'UnitPrice', 'TotalPrice', 'CustomerTotalSpend']
df[numeric_cols].corr()

,Quantity,UnitPrice,TotalPrice,CustomerTotalSpend
Quantity,1.000000,-0.001235,0.886681,0.035980
UnitPrice,-0.001235,1.000000,-0.162029,0.004616
TotalPrice,0.886681,-0.162029,1.000000,0.041800
CustomerTotalSpend,0.035980,0.004616,0.041800,1.000000


## 12. Feature Extraction

Feature extraction derives structured, informative features from **complex or unstructured** data — text, dates, images, etc. — often reducing dimensionality while preserving signal.

In our dataset, Description is free text. Simple extraction examples:
- Word count
- Presence of certain keywords (e.g., "SET", "CHRISTMAS")
- Length of the description string

More advanced extraction (TF-IDF, embeddings) is covered in a dedicated text-features notebook.

In [16]:
df['Description'] = df['Description'].fillna('')
df['DescriptionWordCount'] = df['Description'].apply(lambda x: len(str(x).split()))
df['IsChristmasItem'] = df['Description'].str.contains('CHRISTMAS', case=False, na=False)
df[['Description', 'DescriptionWordCount', 'IsChristmasItem']].head()

,Description,DescriptionWordCount,IsChristmasItem
0,WHITE HANGING HEART T-LIGHT HOLDER,5,False
1,WHITE METAL LANTERN,3,False
2,CREAM CUPID HEARTS COAT HANGER,5,False
3,KNITTED UNION FLAG HOT WATER BOTTLE,6,False
4,RED WOOLLY HOTTIE WHITE HEART.,5,False


## 13. Feature Leakage

**Feature leakage** occurs when a feature contains information that would not be available at prediction time, or that directly encodes the target — causing artificially high performance during training that collapses in production.

Example in our dataset: if we were building a model to predict whether a transaction **is a return** (IsReturn), then:
- Using Quantity directly is *fine* (it's the raw input the flag was derived from — but be careful, since IsReturn = Quantity < 0 makes Quantity a **direct leak** of the target).
- Using CustomerTotalSpend computed **after** including the current transaction is a subtler leak — it uses information (the total including this row) that indirectly encodes outcomes tied to this same row.

**Rule of thumb:** Ask "would I have known this value *before* the event I'm predicting actually happened?" If not, it's leakage.

In [17]:
customer_spend_excl_self = (
    df.groupby('CustomerID')['TotalPrice'].transform('sum') - df['TotalPrice']
)
df['CustomerSpend_ExclCurrent'] = customer_spend_excl_self
df[['CustomerID', 'TotalPrice', 'CustomerTotalSpend', 'CustomerSpend_ExclCurrent']].head()

,CustomerID,TotalPrice,CustomerTotalSpend,CustomerSpend_ExclCurrent
0,17850.0,15.30,5288.63,5273.33
1,17850.0,20.34,5288.63,5268.29
2,17850.0,22.00,5288.63,5266.63
3,17850.0,20.34,5288.63,5268.29
4,17850.0,20.34,5288.63,5268.29


## Role of Feature Engineering in the ML Pipeline

Feature engineering is not a side step — it is the bridge between **raw, messy reality** and a **model that can actually learn** from it.